# S3 bis : E0 rejouable et diagnostics demandes par l'expertise du 17 novembre

Ce notebook repond aux demandes avant E1 : rejouer E0 de bout en bout avec les memes resultats, sauvegarder les scores un par un, corriger la taille de la regression logistique, mesurer le top K par jour avec le bon nombre d'alertes, regarder la calibration, et isoler l'effet du solde par une ablation controlee.

A la fin on a un manifeste (versions, empreinte des donnees, empreintes des sorties) et une note `docs/note_s3_E0.md` regeneree.

In [ ]:
!git clone -q https://github.com/kenzaez/pfa06-graph-fraud.git
%cd pfa06-graph-fraud
!pip install -q -r requirements.txt

## 1. Donnees

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

In [ ]:
!kaggle datasets download -d ealaxi/paysim1 -p data
!unzip -o -q data/paysim1.zip -d data

In [ ]:
from src.audit import load, sha256

PATH = "data/PS_20174392719_1491204439457_log.csv"
assert sha256(PATH) == "16910f90577b0d981bf8ff289714510bb89bc71bff7d3f220f024e287e4eea6b"
df = load(PATH)
df.shape

## 2. Tests du protocole

In [ ]:
!python -m pytest -q tests

## 3. Rejeu complet de E0

On garde d'abord une copie des resultats deja publies dans le depot, puis on relance toute la chaine (grille, choix sur validation, test). Si le code est bien rejouable, les AP doivent etre les memes.

In [ ]:
import shutil
import pandas as pd

shutil.copy("reports/e0/resultats_e0.csv", "/content/resultats_publies.csv")
publies = pd.read_csv("/content/resultats_publies.csv")

In [ ]:
from src import e0

tab, pj, choix = e0.main(df)
choix

In [ ]:
cles = ["modele", "scenario", "population", "partition"]
a = publies.set_index(cles).ap
b = tab.set_index(cles).ap
comp = pd.DataFrame({"ap_publiee": a, "ap_rejeu": b})
comp["ecart"] = (comp.ap_publiee - comp.ap_rejeu).abs()
comp.ecart.max(), comp.sort_values("ecart", ascending=False).head(6)

## 4. Rejeu independant a partir des reglages choisis

On refait les entrainements finaux dans un dossier a part avec les reglages de `config_choisie.json`, puis on compare les scores un par un avec ceux sauvegardes.

In [ ]:
from src import rejeu

r = rejeu.rejouer(df, "/content/rejeu_e0")
r

## 5. Diagnostics demandes

In [ ]:
diag = rejeu.diagnostics()
diag["alertes_seuil"][diag["alertes_seuil"].population == "population_principale"].round(4)

Le seuil gele et le top K par jour sont deux politiques differentes. Le tableau ci dessus donne le nombre d'alertes que le seuil gele emet vraiment chaque jour sur la validation et le test.

In [ ]:
tq = diag["topk_quotidien"]
tq[(tq.population == "population_principale") & (tq.partition == "test") & (tq.scenario == "B") & (tq.modele == "XGBoost") & (tq.K == 250)].round(3)

In [ ]:
ag = diag["topk_agrege"]
ag[(ag.population == "population_principale") & (ag.partition == "test")].round(4)

Meme chose en retirant le jour 31 (aucun negatif ce jour la) :

In [ ]:
hj = diag["topk_hors_jour_sans_negatif"]
hj[(hj.population == "population_principale") & (hj.partition == "test")].round(4)

Le jour 25 est partiel (les steps 595 a 600 seulement) et le jour 31 est partiel et ne contient que des fraudes. La colonne jour_partiel et la colonne jour_sans_negatif de `topk_quotidien.csv` les marquent.

In [ ]:
s1 = diag["sans_un_jour"]
p = s1[(s1.population == "population_principale") & (s1.modele == "XGBoost")].pivot_table(index="jour_retire", columns="scenario", values="ap")
p.round(4)

## 6. Calibration : score moyen contre taux observe

In [ ]:
c = diag["calibration"]
c[(c.population == "population_principale")].round(4)

Un score moyen plus grand que le taux observe veut dire que le modele surestime en moyenne, plus petit qu'il sous estime. Le Brier seul ne donne pas ce sens.

## 7. Ablation controlee du solde

Meme echantillon d'entrainement, meme graine, meme modele, aucune recherche d'hyperparametres. On ajoute le solde puis ses derives, un groupe apres l'autre.

In [ ]:
from src import ablation

abl = ablation.lancer(df)
abl.round(4)

## 8. Tailles et couts apres correction

In [ ]:
res = pd.read_csv("reports/e0/resultats_e0.csv")
res[(res.partition == "test") & (res.population == "population_principale")][["modele", "scenario", "temps_fit_s", "latence_ms_1000tx", "taille_modele_ko"]].round(2)

## 9. Manifeste et note regeneree

In [ ]:
m = rejeu.manifeste(PATH)
{k: m[k] for k in ["python", "pandas", "numpy", "scikit_learn", "xgboost", "commit_git", "donnees_sha256", "graine"]}

In [ ]:
!python -m src.rapport_s3

## 10. Ce qu'il faut recuperer

Telecharger le dossier `reports/` et le dossier `models/` (zip), et remplacer ceux du depot. Les nouveaux fichiers sont `predictions_*.csv.gz`, `topk_quotidien.csv`, `ap_sans_un_jour_test.csv`, `calibration_resume.csv`, `alertes_au_seuil_gele.csv`, `rejeu_e0.csv`, `ablation_solde.csv` et `reports/manifest_e0.json`.

In [ ]:
!zip -q -r sorties_s3bis.zip reports models docs/note_s3_E0.md
from google.colab import files
files.download("sorties_s3bis.zip")